# 06. Cross-Subject Generalization - NinaPro DB1
**Objective:** Evaluate the robustness of classical and deep learning architectures in an inter-subject environment using Leave-One-Subject-Out (LOSO) cross-validation.
 
**Protocol:**
* Training Pool: Subjects 1, 2, 3 (Universal representations)
* Testing Target: Subject 4 (Completely unseen anatomy/electrode placement)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../'))
from src.utils import load_ninapro_mat, extract_zip
from src.preprocess import sEMGPreprocessor, save_preprocessed_hdf5
from src.features import extract_hudgins_features
from src.classification import train_svm_classifier, evaluate_classifier, sEMG1DCNN, train_pytorch_model
from src.dictionary import ChannelWiseDictionaryLearner, ChannelWiseOMPExtractor
from src.config import PREPROCESSED_DIR, DEVICE

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Automated Multi-Subject Preprocessing Pipeline
Ensures Subjects 1 to 4 are extracted, filtered, windowed, balanced, and serialized.

In [2]:
db_dir = "../data/raw/Ninapro_DB1"
exercises = ['E1', 'E2', 'E3']
offsets = {'E1': 0, 'E2': 12, 'E3': 29} 
target_subjects = [1, 2, 3, 4]

preprocessor = sEMGPreprocessor(database_name="DB1")

for sub in target_subjects:
    h5_path = os.path.join(PREPROCESSED_DIR, f"DB1_subject_{sub}.h5")
    if os.path.exists(h5_path):
        print(f"Subject {sub} already preprocessed. Skipping.")
        continue
        
    print(f"\n--- Processing Subject {sub} ---")
    zip_path = os.path.join(db_dir, f"s{sub}.zip")
    extract_dir = os.path.join(db_dir, f"s{sub}")
    
    if not os.path.exists(extract_dir) or len(os.listdir(extract_dir)) == 0:
        extract_zip(zip_path, extract_dir)
        
    emg_list, labels_list, reps_list = [], [], []
    for ex in exercises:
        mat_path = os.path.join(extract_dir, f"S{sub}_A1_{ex}.mat")
        if not os.path.exists(mat_path):
            mat_path = os.path.join(extract_dir, f"s{sub}", f"S{sub}_A1_{ex}.mat")
            
        data = load_ninapro_mat(mat_path)
        labels = data['labels'].copy()
        active_mask = labels > 0
        labels[active_mask] += offsets[ex]
        
        emg_list.append(data['emg'])
        labels_list.append(labels)
        reps_list.append(data['reps'])
        
    emg_full = np.vstack(emg_list)
    labels_full = np.concatenate(labels_list)
    reps_full = np.concatenate(reps_list)
    
    print("Filtering and standardizing...")
    filtered_emg = preprocessor.filter_signal(emg_full)
    norm_emg = preprocessor.standardize(filtered_emg)
    
    print("Extracting and balancing windows...")
    X_win, y_win, reps_win = preprocessor.extract_windows(norm_emg, labels_full, reps_full)
    X_bal, y_bal, reps_bal = preprocessor.balance_rest_class(X_win, y_win, reps_win)
    
    save_preprocessed_hdf5(subject_id=sub, X=X_bal, y=y_bal, reps=reps_bal, db_name="DB1")
    print(f"Subject {sub} serialized successfully.")

Subject 1 already preprocessed. Skipping.

--- Processing Subject 2 ---
Extracting ../data/raw/Ninapro_DB1/s2.zip to ../data/raw/Ninapro_DB1/s2...
Filtering and standardizing...
Extracting and balancing windows...
Subject 2 serialized successfully.

--- Processing Subject 3 ---
Extracting ../data/raw/Ninapro_DB1/s3.zip to ../data/raw/Ninapro_DB1/s3...
Filtering and standardizing...
Extracting and balancing windows...
Subject 3 serialized successfully.

--- Processing Subject 4 ---
Extracting ../data/raw/Ninapro_DB1/s4.zip to ../data/raw/Ninapro_DB1/s4...
Filtering and standardizing...
Extracting and balancing windows...
Subject 4 serialized successfully.


### 2. Data Pooling (Leave-One-Subject-Out)
We aggregate Subjects 1, 2, and 3 for the training pool, and isolate Subject 4 for evaluation.

In [3]:
def load_subject_data(sub_id: int):
    path = os.path.join(PREPROCESSED_DIR, f"DB1_subject_{sub_id}.h5")
    with h5py.File(path, 'r') as f:
        return np.array(f['X']), np.array(f['y'])

# Load and concatenate Training Pool
X_train_list, y_train_list = [], []
for sub in [1, 2, 3]:
    X_sub, y_sub = load_subject_data(sub)
    X_train_list.append(X_sub)
    y_train_list.append(y_sub)

X_train_pool = np.concatenate(X_train_list)
y_train_pool = np.concatenate(y_train_list)

# Load Unseen Testing Target
X_test_unseen, y_test_unseen = load_subject_data(4)

print(f"Universal Training Pool Shape: {X_train_pool.shape}")
print(f"Unseen Test Subject Shape: {X_test_unseen.shape}")
num_classes = len(np.unique(y_train_pool))

Universal Training Pool Shape: (57161, 20, 10)
Unseen Test Subject Shape: (21684, 20, 10)


### 3. Branch A: Universal Hudgins Baseline
Evaluating how standard physical time-domain characteristics transfer across different human anatomies.

In [4]:
print("\n--- Branch A: Hudgins Features + SVM ---")
X_train_hud = extract_hudgins_features(X_train_pool)
X_test_hud = extract_hudgins_features(X_test_unseen)

svm_hud = train_svm_classifier(X_train_hud, y_train_pool, kernel="rbf", C=1.0)
results_hud = evaluate_classifier(svm_hud, X_test_hud, y_test_unseen)

print(f"Hudgins LOSO Accuracy: {results_hud['accuracy'] * 100:.2f}%")
print(f"Hudgins LOSO Macro F1: {results_hud['macro_f1'] * 100:.2f}%")


--- Branch A: Hudgins Features + SVM ---
Extracting Hudgins features from shape (57161, 20, 10)...
Feature extraction complete. Output shape: (57161, 50)
Extracting Hudgins features from shape (21684, 20, 10)...
Feature extraction complete. Output shape: (21684, 50)
Training SVM (Kernel: rbf) on 57161 samples...
Training complete.
Evaluating model on test set...
Hudgins LOSO Accuracy: 8.09%
Hudgins LOSO Macro F1: 6.63%


### 4. Branch C: Universal Deep Learning (1D-CNN)
Evaluating if the PyTorch CNN can learn shift-invariant, subject-agnostic filters.

In [5]:
print("\n--- Branch C: 1D-CNN (End-to-End) ---")
num_channels = X_train_pool.shape[2]

cnn_model = sEMG1DCNN(num_channels=num_channels, num_classes=num_classes)
cnn_model = train_pytorch_model(
    model=cnn_model, 
    X_train=X_train_pool, 
    y_train=y_train_pool, 
    X_test=X_test_unseen, 
    y_test=y_test_unseen,
    epochs=30, # Reduced slightly for pooled dataset size
    batch_size=128, # Increased batch size for the larger dataset
    lr=0.001
)

cnn_model.eval()
with torch.no_grad():
    X_te_tensor = torch.FloatTensor(X_test_unseen).transpose(1, 2).to(DEVICE)
    raw_outputs = cnn_model(X_te_tensor)
    cnn_preds = torch.argmax(raw_outputs, dim=1).cpu().numpy()

from sklearn.metrics import accuracy_score, f1_score
cnn_accuracy = accuracy_score(y_test_unseen, cnn_preds)
cnn_f1 = f1_score(y_test_unseen, cnn_preds, average='macro')

print(f"1D-CNN LOSO Accuracy: {cnn_accuracy * 100:.2f}%")
print(f"1D-CNN LOSO Macro F1: {cnn_f1 * 100:.2f}%")


--- Branch C: 1D-CNN (End-to-End) ---
Training on cuda...
Epoch 1/30 | Loss: 3.6009
Epoch 5/30 | Loss: 2.7731
Epoch 10/30 | Loss: 2.5767
Epoch 15/30 | Loss: 2.4752
Epoch 20/30 | Loss: 2.4150
Epoch 25/30 | Loss: 2.3695
Epoch 30/30 | Loss: 2.3204
1D-CNN LOSO Accuracy: 8.74%
1D-CNN LOSO Macro F1: 7.91%


### 5. Branch B: Universal Dictionary Learning
If intra-subject Dictionary Classification failed due to spatial overfitting, does training on multiple subjects force it to learn a more generalized, subject-agnostic basis?

In [6]:
print("\n--- Branch B: Universal Dictionary Features ---")
learner_univ = ChannelWiseDictionaryLearner(n_atoms=64, n_nonzero_coefs=5)
# Sub-sampling the pool to fit in RAM for Dictionary Learning if necessary
sample_idx = np.random.choice(X_train_pool.shape[0], 20000, replace=False)
learner_univ.fit(X_train_pool[sample_idx])

extractor_univ = ChannelWiseOMPExtractor(learner_univ.dictionary_, n_nonzero_coefs=5)
X_train_omp = extractor_univ.transform(X_train_pool)
X_test_omp = extractor_univ.transform(X_test_unseen)

scaler = StandardScaler()
X_train_omp_scaled = scaler.fit_transform(X_train_omp)
X_test_omp_scaled = scaler.transform(X_test_omp)

svm_omp = train_svm_classifier(X_train_omp_scaled, y_train_pool, kernel="rbf", C=1.0)
results_omp = evaluate_classifier(svm_omp, X_test_omp_scaled, y_test_unseen)

print(f"Universal Dictionary LOSO Accuracy: {results_omp['accuracy'] * 100:.2f}%")
print(f"Universal Dictionary LOSO Macro F1: {results_omp['macro_f1'] * 100:.2f}%")


--- Branch B: Universal Dictionary Features ---
Training 1D Dictionary with 64 atoms on 200000 temporal windows...
Channel-wise dictionary learning complete.


/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


Training SVM (Kernel: rbf) on 57161 samples...
Training complete.
Evaluating model on test set...
Universal Dictionary LOSO Accuracy: 5.91%
Universal Dictionary LOSO Macro F1: 3.28%
